In [0]:
# Helper functions for Bronze/Silver/Gold layer processing
# These functions accept catalog and schema names as parameters
# Call them from your notebooks (bronze_work, silver_work, gold_work) with appropriate values

from pyspark.sql import functions as F
from datetime import datetime
from delta import DeltaTable

## Helper Functions for ELT Pipeline

This notebook contains reusable functions for Bronze, Silver, and Gold layer processing.

### Usage
1. Import this notebook using `%run "../notebooks/helper_functions"`
2. Create widgets in your calling notebook (bronze_work, silver_work, gold_work)
3. Pass catalog/schema names as parameters to each function

### Function Categories

#### Bronze Functions
- `get_last_successfull_watermark(table_name, control_table)` - Get last watermark from control table
- `upsert_bronze_control(...)` - Update bronze control table

#### Silver Functions
- `upsert_to_silver(df_source, target_table, join_keys)` - Merge data to silver table
- `get_last_processed_bronze_ingested_at(entity_name, control_table)` - Get silver watermark
- `upsert_silver_control(...)` - Update silver control table
- `get_incremental_bronze(bronze_table, entity_name, control_table)` - Read incremental bronze data

#### Gold Functions
- `upsert_to_gold(df_source, target_table, join_key)` - Merge data to gold table
- `get_last_processed_silver_ts(entity_name, control_table)` - Get gold watermark
- `upsert_gold_control(...)` - Update gold control table

#### Bronze layer ingestion
- get_last_successful_watermark() - reads the last processed watermark from the control table
- upsert_bronze_control() - updatest the control table after a successful Bronze Load

In [0]:
def get_last_successfull_watermark(table_name: str, control_table: str):
    """
    Get last successful watermark from bronze control table.
    
    Args:
        table_name: Name of the source table
        control_table: Full name of control table (e.g., 'catalog.schema.ingestion_control')
    
    Returns:
        Tuple of (last_successful_ts, last_successful_pk)
    """
    ctrl = (
        spark.table(control_table)
        .filter(
            (F.col("layer") == "bronze") &
            (F.col("table_name") == table_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None, None
    last_successful_ts = rows[0]["last_successful_ts"]
    last_successful_pk = rows[0]["last_successful_pk"]
    return last_successful_ts, last_successful_pk

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id, control_table: str):
    """
    Update bronze control table with ingestion metadata.
    
    Args:
        table_name: Name of the source table
        ts_col: Timestamp column name
        pk_col: Primary key column name
        last_ts: Last processed timestamp
        last_pk: Last processed primary key
        rows_written: Number of rows written
        run_id: Bronze run ID
        control_table: Full name of control table (e.g., 'catalog.schema.ingestion_control')
    """
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.utcnow()
        )],
        schema = """
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )
    dt = DeltaTable.forName(spark, control_table)
    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            F.expr("t.table_name = s.table_name and t.layer = s.layer")
        )
        .whenMatchedUpdate(
            set={
                "ts_col" : F.col("s.ts_col"),
                "pk_col" : F.col("s.pk_col"),
                "last_successful_ts" : F.col("s.last_successful_ts"),
                "last_successful_pk" : F.col("s.last_successful_pk"),
                "last_run_id" : F.col("s.last_run_id"),
                "rows_written" : F.col("s.rows_written"),
                "run_status" : F.col("s.run_status"),
                "updated_at" : F.col("s.updated_at")
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

#### Silver Layer Ingestion 
This cell contains reusable logic for Silver
- upsert_to_silver() - merges cleaned / transformed rows into the Silver target table
- get_last_processed_bronze_ingested_at() - reads the silver watermark
- upsert_silver_control() - updates the silver control table
- get_incremental_bronze() - reads only new Bronze rows that Silver has not processed yet

In [0]:
def upsert_to_silver(df_source, target_table, join_keys):
    if spark.catalog.tableExists(target_table):
        dynamic_join_condition = " AND ".join([f"source.{c} = target.{c}" for c in join_keys])
        print(f"Merge condition for {target_table} is {dynamic_join_condition}")
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("target")
            .merge(
                df_source.alias("source"),
                F.expr(dynamic_join_condition)
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str, control_table: str):
    """
    Get last processed bronze timestamp from silver control table.
    
    Args:
        entity_name: Name of the silver entity
        control_table: Full name of control table (e.g., 'catalog.schema.processing_control')
    
    Returns:
        Tuple of (last_processed_bronze_ingested_at, last_processed_bronze_run_id)
    """
    ctrl = (
        spark.table(control_table)
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None, None

    return rows[0]["last_processed_bronze_ingested_at"], rows[0]["last_processed_bronze_run_id"]

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged, silver_run_id, control_table: str):
    """
    Update silver control table with processing metadata.
    
    Args:
        entity_name: Name of the silver entity
        last_processed_bronze_run_id: Last bronze run ID processed
        last_processed_bronze_ingested_at: Last bronze ingestion timestamp processed
        rows_merged: Number of rows merged
        silver_run_id: Current silver run ID
        control_table: Full name of control table (e.g., 'catalog.schema.processing_control')
    """
    control_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "success",
            datetime.utcnow(),
            silver_run_id            
        )],
        schema = """
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,       
        updated_at timestamp,
        silver_run_id string
        """
    )
    dt = DeltaTable.forName(spark, control_table)
    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            F.expr("t.entity_name = s.entity_name and t.layer = s.layer")
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id" : F.col("s.last_processed_bronze_run_id"),
                "last_processed_bronze_ingested_at" : F.col("s.last_processed_bronze_ingested_at"),
                "rows_merged" : F.col("s.rows_merged"),
                "run_status" : F.col("s.run_status"),
                "updated_at" : F.col("s.updated_at"),
                "silver_run_id": F.col("s.silver_run_id")
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table, entity_name, control_table: str):
    """
    Get incremental bronze data that hasn't been processed by silver yet.
    
    Args:
        bronze_table: Full name of bronze table
        entity_name: Name of the silver entity
        control_table: Full name of silver control table
    
    Returns:
        Tuple of (incremental_df, last_ingested_at, last_run_id)
    """
    last_ingested_at, last_run_id = get_last_processed_bronze_ingested_at(entity_name, control_table)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at, last_run_id

    bronze_latest_records_df = bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at))

    return bronze_latest_records_df, last_ingested_at, last_run_id

#### Gold Layer Ingestion 
This cell contains reusable logic for Gold functions
- upsert_to_gold() - merges data rows into Gold current-state table
- get_last_processed_silver_ts() - reads the Gold watermark from the control table
- upsert_gold_control() - updates Gold control after a successful run

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
           .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute()
        )
    else:
        df_source.write.format("delta").saveAsTable(target_table)


In [0]:
def get_last_processed_silver_ts(entity_name: str, control_table: str):
    """
    Get last processed silver timestamp from gold control table.
    
    Args:
        entity_name: Name of the gold entity
        control_table: Full name of control table (e.g., 'catalog.schema.processing_control')
    
    Returns:
        Last processed silver run timestamp or None
    """
    ctrl = (
        spark.table(control_table)
             .filter(
                 (F.col("layer") == 'gold') &
                 (F.col("entity_name") == entity_name) &
                 (F.col("run_status") == 'SUCCESS')
             )
             .orderBy(F.col("updated_at").desc())
             .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_run_ts, rows_merged, gold_run_id, control_table: str):
    """
    Update gold control table with processing metadata.
    
    Args:
        entity_name: Name of the gold entity
        last_processed_silver_run_id: Last silver run ID processed
        last_processed_run_ts: Last silver run timestamp processed
        rows_merged: Number of rows merged
        gold_run_id: Current gold run ID
        control_table: Full name of control table (e.g., 'catalog.schema.processing_control')
    """
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_run_ts,
            int(rows_merged),
            "SUCCESS",
            gold_run_id,
            datetime.utcnow()
        )],
        schema = """
        layer string,
        entity_name string,
        last_processed_silver_run_id string,
        last_processed_silver_run_ts timestamp,
        rows_merged bigint,
        run_status string,
        gold_run_id string,
        updated_at timestamp
        """
    )
    dt = DeltaTable.forName(spark, control_table)
    (dt.alias("t")
        .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name")
        .whenMatchedUpdate(set={
            "last_processed_silver_run_id" : "s.last_processed_silver_run_id",
            "last_processed_silver_run_ts" : "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute())
    